# FLUX.1-schnell style test (L4)

A standalone test rig for **Lightning AI's L4** box -- separate from `notebooks/studio.ipynb`
(the Kaggle T4 x2 / Krea 2 production notebook, preserved untouched on the `krea2` branch).
This does **not** touch that pipeline. Point of this notebook: find out, cheaply, whether
FLUX.1-schnell is actually faster than Krea2Turbo was and which house-style block (the
original Whymentary monoline stickman, or the flat-vector pivot picked for Krea 2) renders
better on this model, before committing to a rewrite of the production server.

**Before running:** FLUX.1-schnell is Apache-2.0 (unrestricted, commercial-fine), but the
HF repo is still gated for download tracking -- accept the license once at
https://huggingface.co/black-forest-labs/FLUX.1-schnell, then set `HF_TOKEN` below (or run
`huggingface-cli login` in a terminal first).

**Worth knowing going in:** the L4 is a 72W-TDP inference card -- basically the same power
class as the T4 that throttled Krea 2, just a newer/more efficient architecture (Ada,
24GB, native bf16). It may still power-throttle under sustained load; this notebook times
every generation so that shows up in the numbers instead of being assumed away.

In [ ]:
import os, time, pathlib

!pip install -q -U diffusers transformers accelerate sentencepiece protobuf huggingface_hub

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# HF_TOKEN env var, or fall back to an interactive login prompt.
from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    login()  # pastes a token prompt -- needs the FLUX.1-schnell license accepted on HF first

In [ ]:
from diffusers import FluxPipeline

# bf16 is native on the L4 (Ada) -- no T4-style fp16 patching needed here.
# enable_model_cpu_offload() keeps only the active submodule resident on GPU (streams the
# rest from host RAM), which is the standard way to fit the ~33GB full bf16 pipeline
# (12B transformer + T5-XXL text encoder + CLIP + VAE) onto a single 24GB card. Costs some
# speed vs. everything resident at once, but avoids OOM as the first thing to try; if the
# timings below look offload-bound rather than compute-bound, that's the lever to revisit.
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
print("loaded")

In [ ]:
# Two style blocks from AGENTS.md, head-to-head -- same comparison that picked flat-vector
# for Krea 2. Don't assume the winner carries over: it won there because of a Krea-specific
# anatomy failure on monoline, not because flat-vector is universally better.
STYLES = {
    "monoline": ("minimal hand-drawn stickman on a pure white background, thick 6px solid "
                 "black monoline ink strokes, flat saturated color fills, no gradients, no "
                 "shadows, whiteboard explainer illustration"),
    "flat_vector": ("MS Paint style drawing, thin uneven black outline, flat bucket-fill "
                    "saturated colors, no shading, no gradients, naive simple "
                    "computer-paint-program look"),
}

# Subjects from NICHE.md's AI-mechanical-analogy starter kit -- real candidate content, not
# throwaway test prompts, so a good result here is directly reusable.
SUBJECTS = {
    "next_token": "a crank-operated fortune-teller machine spitting a word out of a slot-machine reel, no brain inside, just gears matching patterns",
    "hallucination": "a broken photocopier confidently printing garbage pages as if they were the original",
    "training_data": "a giant funnel pouring a torrent of documents and web pages into a grinder",
}

PROMPTS = [(s_name, sub_name, f"{s_body}. {sub_body}")
           for s_name, s_body in STYLES.items()
           for sub_name, sub_body in SUBJECTS.items()]
print(f"{len(PROMPTS)} generations queued")

In [ ]:
from IPython.display import display

OUT = pathlib.Path("shots/flux_schnell_test")  # gitignored like every other shots/ subdir
OUT.mkdir(parents=True, exist_ok=True)

results = []
for style_name, subject_name, prompt in PROMPTS:
    t0 = time.time()
    image = pipe(
        prompt,
        guidance_scale=0.0,          # schnell is guidance-free, same as Krea2Turbo's guide_scale=0.0
        num_inference_steps=4,       # schnell's native step count -- don't push this up, it wasn't distilled for more
        max_sequence_length=256,     # schnell's supported max (dev uses 512)
        height=576, width=1024,      # matches the production notebook's shot aspect ratio
        generator=torch.Generator("cpu").manual_seed(7),
    ).images[0]
    secs = round(time.time() - t0, 1)
    name = f"{style_name}_{subject_name}.png"
    image.save(OUT / name)
    results.append({"name": name, "style": style_name, "subject": subject_name, "s": secs})
    print(f"[{secs:>5.1f}s] {name}")
    display(image)

In [ ]:
import statistics

secs = [r["s"] for r in results]
print(f"n={len(secs)}  mean={statistics.mean(secs):.1f}s  min={min(secs):.1f}s  max={max(secs):.1f}s")
print(f"\nFor comparison, Krea2Turbo on Kaggle T4 x2: ~55-60s/image at 8 steps (power-throttled).")
print(f"Images saved to {OUT}/ -- eyeball monoline_* vs flat_vector_* per subject before deciding a winner.")